First AHL-D pass (older custom split, 21326 train / 2585 val). Superseded by 02_ahld_crnn_baseline.

In [ ]:
#@title Download + Prepare AHL-D 29K

!pip install -q datasets jiwer

import os
import json
import numpy as np
import pandas as pd

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset


ahld = load_dataset(
    "misiker/AHLD-29k"
)

print(ahld)

print(
    f"Train samples: {len(ahld['train'])}"
)

if "validation" in ahld:
    print(
        f"Validation samples: {len(ahld['validation'])}"
    )

if "test" in ahld:
    print(
        f"Test samples: {len(ahld['test'])}"
    )


image_column = None
text_column = None

for column in ahld["train"].column_names:

    if column.lower() in [
        "image",
        "img",
        "picture"
    ]:
        image_column = column
        break


for column in ahld["train"].column_names:

    if column.lower() in [
        "text",
        "label",
        "transcription",
        "transcript",
        "sentence",
        "gt"
    ]:
        text_column = column
        break


if image_column is None:
    raise RuntimeError(
        f"Image column not found: "
        f"{ahld['train'].column_names}"
    )

if text_column is None:
    raise RuntimeError(
        f"Text column not found: "
        f"{ahld['train'].column_names}"
    )


train_df = pd.DataFrame(
    ahld["train"]
)


if "validation" in ahld:

    val_df = pd.DataFrame(
        ahld["validation"]
    )

elif "test" in ahld:

    full_df = pd.DataFrame(
        ahld["train"]
    )

    val_size = int(
        len(full_df) * 0.1
    )

    val_df = full_df.iloc[
        -val_size:
    ].reset_index(drop=True)

    train_df = full_df.iloc[
        :-val_size
    ].reset_index(drop=True)

else:

    raise RuntimeError(
        "AHL-D has no validation or test split."
    )


IMG_HEIGHT = 64
IMG_WIDTH = 512
BATCH_SIZE = 16


all_text = (
    train_df[text_column]
    .astype(str)
    .tolist()
)


characters = sorted(
    set(
        "".join(all_text)
    )
)


char_to_idx = {
    char: idx
    for idx, char in enumerate(characters)
}


idx_to_char = {
    idx: char
    for char, idx in char_to_idx.items()
}


blank_idx = len(characters)


class AHLDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(
        self,
        idx
    ):

        row = self.dataframe.iloc[idx]

        image = row[image_column]

        text = str(
            row[text_column]
        )

        if not isinstance(
            image,
            Image.Image
        ):

            image = Image.fromarray(
                np.asarray(image)
            )

        image = image.convert("L")

        image = image.resize(
            (
                IMG_WIDTH,
                IMG_HEIGHT
            ),
            Image.Resampling.LANCZOS
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        target = torch.tensor(
            [
                char_to_idx[c]
                for c in text
                if c in char_to_idx
            ],
            dtype=torch.long
        )

        return (
            image,
            target,
            len(target)
        )


def collate_fn(batch):

    images = torch.stack(
        [
            item[0]
            for item in batch
        ]
    )

    targets = torch.cat(
        [
            item[1]
            for item in batch
        ]
    )

    target_lengths = torch.tensor(
        [
            item[2]
            for item in batch
        ],
        dtype=torch.long
    )

    return (
        images,
        targets,
        target_lengths
    )


train_dataset = AHLDataset(
    train_df
)

val_dataset = AHLDataset(
    val_df
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn
)


print(
    f"AHL-D ready | "
    f"train={len(train_dataset)} | "
    f"validation={len(val_dataset)}"
)

print(
    f"Vocabulary size: {len(characters)}"
)

In [ ]:
#@title AHL-D 29K CRNN

STAGE_ROOT = f"{OCR_RESEARCH_ROOT}/crnn/ahld29k"

os.makedirs(
    STAGE_ROOT,
    exist_ok=True
)


MAX_EPOCHS = 20
LEARNING_RATE = 1e-3
PATIENCE = 4
NUM_SAMPLE_PREDICTIONS = 3


CONFIG_PATH = f"{STAGE_ROOT}/config.json"
HISTORY_PATH = f"{STAGE_ROOT}/epoch_history.csv"
BEST_PATH = f"{STAGE_ROOT}/best_checkpoint.pt"
LAST_PATH = f"{STAGE_ROOT}/last_checkpoint.pt"
METRICS_PATH = f"{STAGE_ROOT}/metrics.json"
PREDICTIONS_PATH = f"{STAGE_ROOT}/predictions.csv"
SAMPLES_PATH = f"{STAGE_ROOT}/sample_predictions.txt"
README_PATH = f"{STAGE_ROOT}/README.md"


class CRNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                1, 64, 3, padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                2, 2
            ),


            nn.Conv2d(
                64, 128, 3, padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                2, 2
            ),


            nn.Conv2d(
                128, 256, 3, padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                256, 256, 3, padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=(2, 1),
                stride=(2, 1)
            ),


            nn.Conv2d(
                256, 512, 3, padding=1
            ),

            nn.BatchNorm2d(512),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                512, 512, 3, padding=1
            ),

            nn.BatchNorm2d(512),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=(2, 1),
                stride=(2, 1)
            )
        )


        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.2
        )


        self.classifier = nn.Linear(
            512,
            num_classes
        )


    def forward(self, x):

        x = self.cnn(x)

        x = F.adaptive_avg_pool2d(
            x,
            (1, x.size(3))
        )

        x = x.squeeze(2)

        x = x.permute(
            2,
            0,
            1
        )

        x, _ = self.rnn(x)

        x = self.classifier(x)

        return x


model = CRNN(
    num_classes=len(characters) + 1
).to(DEVICE)


criterion = nn.CTCLoss(
    blank=blank_idx,
    zero_infinity=True
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)


start_epoch = 1
best_cer = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []


if os.path.exists(LAST_PATH):

    checkpoint = torch.load(
        LAST_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scheduler_state_dict" in checkpoint:

        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    start_epoch = (
        checkpoint["epoch"] + 1
    )

    best_cer = checkpoint.get(
        "best_cer",
        float("inf")
    )

    best_epoch = checkpoint.get(
        "best_epoch",
        0
    )

    epochs_without_improvement = checkpoint.get(
        "epochs_without_improvement",
        0
    )

    if os.path.exists(HISTORY_PATH):

        history = pd.read_csv(
            HISTORY_PATH
        ).to_dict("records")

    print(
        f"Resuming | epoch {start_epoch}"
    )

else:

    print(
        "Starting AHL-D CRNN"
    )


config = {
    "dataset": "misiker/AHLD-29k",
    "model": "CRNN",
    "image_height": IMG_HEIGHT,
    "image_width": IMG_WIDTH,
    "vocabulary_size": len(characters),
    "num_classes": len(characters) + 1,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "patience": PATIENCE,
    "optimizer": "AdamW",
    "weight_decay": 1e-4,
    "loss": "CTCLoss",
    "seed": SEED,
    "train_samples": len(train_df),
    "validation_samples": len(val_df)
}


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )


def save_checkpoint(
    path,
    epoch,
    best_cer,
    best_epoch,
    epochs_without_improvement
):

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_cer": best_cer,
            "best_epoch": best_epoch,
            "epochs_without_improvement":
                epochs_without_improvement,
            "char_to_idx": char_to_idx,
            "idx_to_char": idx_to_char,
            "blank_idx": blank_idx,
            "config": config
        },
        path
    )


with open(
    README_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "# AHL-D 29K CRNN Training\n\n"
        f"- Dataset: misiker/AHLD-29k\n"
        f"- Training samples: {len(train_df)}\n"
        f"- Validation samples: {len(val_df)}\n"
        f"- Image size: {IMG_WIDTH}x{IMG_HEIGHT}\n"
        f"- Batch size: {BATCH_SIZE}\n"
        f"- Maximum epochs: {MAX_EPOCHS}\n"
        f"- Optimizer: AdamW\n"
        f"- Learning rate: {LEARNING_RATE}\n"
        f"- Loss: CTCLoss\n"
    )


for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1
):

    model.train()

    running_loss = 0.0
    batch_count = 0


    for images, targets, target_lengths in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        targets = targets.to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(
            images
        )


        input_lengths = torch.full(
            size=(images.size(0),),
            fill_value=outputs.size(0),
            dtype=torch.long,
            device=DEVICE
        )


        loss = criterion(
            outputs.log_softmax(2),
            targets,
            input_lengths,
            target_lengths.to(DEVICE)
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )


        optimizer.step()


        running_loss += loss.item()

        batch_count += 1


    train_loss = (
        running_loss / batch_count
    )


    evaluation = evaluate_model(
        model,
        val_loader
    )


    val_cer = evaluation["cer"]
    val_wer = evaluation["wer"]
    exact_accuracy = evaluation["exact_accuracy"]


    scheduler.step(
        val_cer
    )


    improved = (
        val_cer < best_cer
    )


    if improved:

        best_cer = val_cer
        best_epoch = epoch
        epochs_without_improvement = 0


        save_checkpoint(
            BEST_PATH,
            epoch,
            best_cer,
            best_epoch,
            epochs_without_improvement
        )


        save_predictions(
            evaluation["references"],
            evaluation["predictions"],
            PREDICTIONS_PATH
        )


        sample_lines = []


        for reference, prediction in zip(
            evaluation["references"][:20],
            evaluation["predictions"][:20]
        ):

            sample_lines.append(
                f"GT: {reference}\n"
                f"PR: {prediction}\n"
            )


        with open(
            SAMPLES_PATH,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                "\n".join(sample_lines)
            )


    else:

        epochs_without_improvement += 1


    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_cer": val_cer,
        "val_wer": val_wer,
        "exact_accuracy": exact_accuracy,
        "learning_rate":
            optimizer.param_groups[0]["lr"],
        "best_cer": best_cer,
        "best_epoch": best_epoch
    }


    history.append(
        epoch_record
    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


    save_checkpoint(
        LAST_PATH,
        epoch,
        best_cer,
        best_epoch,
        epochs_without_improvement
    )


    save_metrics(
        {
            "dataset":
                "misiker/AHLD-29k",

            "current_epoch":
                epoch,

            "best_epoch":
                best_epoch,

            "best_cer":
                best_cer,

            "current_cer":
                val_cer,

            "current_wer":
                val_wer,

            "current_exact_accuracy":
                exact_accuracy
        },
        METRICS_PATH
    )


    print(
        f"\nEpoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"CER {val_cer:.4f} | "
        f"WER {val_wer:.4f}"
        + (
            " | best"
            if improved
            else ""
        )
    )


    for reference, prediction in zip(
        evaluation["references"][
            :NUM_SAMPLE_PREDICTIONS
        ],
        evaluation["predictions"][
            :NUM_SAMPLE_PREDICTIONS
        ]
    ):

        print(
            f"GT: {reference}"
        )

        print(
            f"PR: {prediction}"
        )


    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            f"Early stop | "
            f"best epoch {best_epoch}"
        )

        break


print(
    f"\nDone | "
    f"best epoch {best_epoch} | "
    f"CER {best_cer:.4f}"
)

print(
    f"Best model: {BEST_PATH}"
)

print(
    f"Last checkpoint: {LAST_PATH}"
)

NameError: name 'OCR_RESEARCH_ROOT' is not defined